In [1]:
!pip install onnx onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 7.3 MB/s eta 0:00:00


In [2]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 11.4 MB/s eta 0:00:00



# BirdCLEF 2026 — Full Pipeline (Offline-safe, GPU/CPU)
We created cpu and offline pipeline for BirdCLEF 2026 comptetion. 
The instruction:
The notebook is optimised to be used in kaggle for competition, so load the notebook on kaggle.
training
Run all cells top to bottom.
 Outputs: bird_sed_model.pth  +  bird_model.onnx
bird_model.onnx.data
bird_sed_model.pth
submission.csv
target_columns.json


Competitions with violations on demand require special requirements: the model must be tested on the CPU, without an Internet connection.

Competitions with violations on demand require special requirements: the model must be tested on the CPU, without an Internet connection.

The task was completed as follows:
1) 12 epochs (we completely wounded the laptop from start to finish).
2) Since the submission did not allow us to throw the submission, we had to throw the entire laptop first so that it passed on the test data and the output that it gave was our submission.
3) It is illogical to wound the model and train it again on the tests + it also gave a Timeout error, so it was decided to separate the laptop and train the inference part separately so that on the test data on the CPU and avoid this problem
4) The execution should be without a network connection, so we decided that it is worth saving the weights of the trained model and using them separately, as a new dataset birdclef-model-v1-zvuk2
5) The experience of last year's competition showed that optimization on the CPU is necessary. We chose to do this through the Open Neural Network Exchange (onnx runtime module). pip-install of this module without a network is not possible properly, so we decided to download the wheels separately, which were found on kaggle as birdclef-2026-download-wheels. There we needed the version:

The code of inference is in notebook:
STEP 2 — INFERENCE / SUBMISSION
test_1.ipynb


In [ ]:



#  CELL 1Imports & reproducibility

import os
import gc
import json
import glob
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import timm

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm


try:
    import onnx
    import onnxruntime as ort
    import onnxscript 
    ONNX_OK = True
except ImportError:
    ONNX_OK = False
    print("[WARN] onnx, onnxruntime, or onnxscript not found.")
    print("       Run: !pip install onnx onnxruntime onnxscript")

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch    : {torch.__version__}")
print(f"torchaudio : {torchaudio.__version__}")
print(f"CUDA       : {torch.cuda.is_available()}")
print(f"ONNX ready : {ONNX_OK}")


# 2 config
class CFG:
    seed = 42

    BASE_DIR            = "/kaggle/input/competitions/birdclef-2026"
    TRAIN_AUDIO_DIR     = f"{BASE_DIR}/train_audio"
    TRAIN_CSV           = f"{BASE_DIR}/train.csv"
    TAXONOMY_CSV        = f"{BASE_DIR}/taxonomy.csv"
    TEST_DIR            = f"{BASE_DIR}/test_soundscapes"
    SAMPLE_SUB          = f"{BASE_DIR}/sample_submission.csv"
    OUTPUT_DIR          = "/kaggle/working"

    PRETRAINED_WEIGHTS_PATH = None

    TRAINED_MODEL_PATH  = f"{OUTPUT_DIR}/bird_sed_model.pth"
    ONNX_PATH           = f"{OUTPUT_DIR}/bird_model.onnx"


    TARGET_SR           = 32_000
    SEGMENT_SEC         = 5.0

    
    N_MELS              = 128
    N_FFT               = 1024
    HOP_LENGTH          = 320        
    FMIN                = 20.0
    FMAX                = 16_000.0


    MODEL_NAME          = "efficientnetv2_s"
    N_FOLDS             = 5
    TRAIN_FOLDS         = [0, 1, 2, 3]   
    VAL_FOLD            = 4
    EPOCHS              = 12
    BATCH_SIZE          = 32
    NUM_WORKERS         = 2
    LR                  = 1e-3
    WEIGHT_DECAY        = 1e-4
    RARE_THRESH         = 20

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    AMP_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


cfg = CFG()
Path(cfg.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Device : {cfg.DEVICE}")
print(f"AMP    : {cfg.AMP_DEVICE}")


# 3  Data loading & target columns

df       = pd.read_csv(cfg.TRAIN_CSV)
taxonomy = pd.read_csv(cfg.TAXONOMY_CSV)


TARGET_COLUMNS = sorted(taxonomy["primary_label"].astype(str).unique().tolist())
NUM_CLASSES    = len(TARGET_COLUMNS)
LABEL2IDX      = {lbl: i for i, lbl in enumerate(TARGET_COLUMNS)}

df["file_path"] = cfg.TRAIN_AUDIO_DIR + "/" + df["filename"].astype(str)
df["primary_label"] = df["primary_label"].astype(str)

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.seed)
df["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(df, df["primary_label"])):
    df.loc[df.index[val_idx], "fold"] = fold_idx

print(f"Total samples  : {len(df):,}")
print(f"Unique species : {NUM_CLASSES}")
print(f"Fold counts    :\n{df['fold'].value_counts().sort_index().to_string()}")


# CELL 4  Dataset

class BirdCLEFDataset(Dataset):
    """
    Loads .ogg clips, converts to log-Mel spectrogram, returns (spec, label).

    All class metadata is passed in at construction — no module-level globals.
    This makes each cell fully self-contained and safe after kernel restarts.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        num_classes: int,
        label2idx: dict,
        mode: str = "train",
        segment_sec: float = CFG.SEGMENT_SEC,
        target_sr: int = CFG.TARGET_SR,
    ):
        self.df          = df.reset_index(drop=True)
        self.num_classes = num_classes
        self.label2idx   = label2idx
        self.mode        = mode
        self.seg_len     = int(segment_sec * target_sr)

        self.mel_transform = T.MelSpectrogram(
            sample_rate=target_sr,
            n_fft=CFG.N_FFT,
            hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS,
            f_min=CFG.FMIN,
            f_max=CFG.FMAX,
        )
        self.db_transform = T.AmplitudeToDB(stype="power", top_db=80)

        self.freq_mask = T.FrequencyMasking(freq_mask_param=15)
        self.time_mask = T.TimeMasking(time_mask_param=35)

    def __len__(self):
        return len(self.df)

    def _load_wave(self, path: str) -> torch.Tensor:
        """Load audio, convert to mono, crop/pad to fixed length."""
        try:
            wav, sr = torchaudio.load(path)
        except Exception:
            # Return silence on corrupt files
            return torch.zeros(1, self.seg_len)

        
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if wav.shape[1] >= self.seg_len:
            if self.mode == "train":
                start = random.randint(0, wav.shape[1] - self.seg_len)
            else:
                start = (wav.shape[1] - self.seg_len) // 2
            wav = wav[:, start : start + self.seg_len]
        else:
            pad = self.seg_len - wav.shape[1]
            wav = F.pad(wav, (0, pad))

        return wav

    def _to_spec(self, wav: torch.Tensor) -> torch.Tensor:
        """Waveform → normalised log-Mel spectrogram (1, N_MELS, T)."""
        spec = self.db_transform(self.mel_transform(wav))   
        mean = spec.mean()
        std  = spec.std() + 1e-6
        return (spec - mean) / std

    def __getitem__(self, idx: int):
        row  = self.df.iloc[idx]
        wav  = self._load_wave(row["file_path"])
        spec = self._to_spec(wav)

        if self.mode == "train":
            spec = self.freq_mask(spec)
            spec = self.time_mask(spec)
            if random.random() < 0.3:
                spec = spec + torch.randn_like(spec) * 0.05

        label = torch.zeros(self.num_classes, dtype=torch.float32)
        lbl   = str(row["primary_label"])
        if lbl in self.label2idx:
            label[self.label2idx[lbl]] = 1.0

        return spec, label


#  5  Model architecture

class BirdSEDModel(nn.Module):

    def __init__(
        self,
        model_name: str,
        num_classes: int,
        pretrained: bool = False,
        pretrained_path: str = None,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,         
            in_chans=1,
            num_classes=0,                  
            global_pool="",                 
        )

        if pretrained_path and Path(pretrained_path).is_file():
            state = torch.load(pretrained_path, map_location="cpu")
            state = {k.replace("backbone.", ""): v for k, v in state.items()}
            missing, unexpected = self.backbone.load_state_dict(state, strict=False)
            print(f"[INFO] Loaded backbone from {pretrained_path}")
            print(f"       Missing keys : {len(missing)} | Unexpected : {len(unexpected)}")


        with torch.no_grad():
            dummy = torch.zeros(1, 1, 128, 312)
            feat  = self.backbone(dummy)
            in_ch = feat.shape[1]           

        self.dropout = nn.Dropout(0.3)
        self.fc_clip = nn.Linear(in_ch, num_classes)   
        self.fc_att  = nn.Linear(in_ch, num_classes)   

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (B, 1, N_MELS, T)
        returns : (B, num_classes)  
        """
        feat = self.backbone(x)            

        feat = feat.mean(dim=2)            
        feat = feat.permute(0, 2, 1)       
        feat = self.dropout(feat)

        clip_logits = self.fc_clip(feat)                        
        att_weights = torch.softmax(self.fc_att(feat), dim=1)  

        out = (torch.sigmoid(clip_logits) * att_weights).sum(dim=1)
        return out  



# 6  Loss function & helpers

class FocalLoss(nn.Module):
    """Binary focal loss for multi-label classification."""

    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, probs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs   = probs.float()
        targets = targets.float()
        
        probs   = torch.clamp(probs, 1e-6, 1 - 1e-6)
        bce     = F.binary_cross_entropy(probs, targets, reduction="none")
        pt      = torch.where(targets == 1, probs, 1 - probs)
        focal_w = self.alpha * (1 - pt) ** self.gamma
        return (focal_w * bce).mean()




def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler, device):
    model.train()
    running_loss = 0.0

    for specs, labels in tqdm(loader, desc="  train", leave=False):
        specs  = specs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)


        with torch.amp.autocast(device_type=cfg.AMP_DEVICE):
            preds = model(specs)
            
        loss = criterion(preds.float(), labels.float())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()

    return running_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for specs, labels in tqdm(loader, desc="    val", leave=False):
            specs  = specs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)


            with torch.amp.autocast(device_type=cfg.AMP_DEVICE):
                preds = model(specs)
                
            loss = criterion(preds.float(), labels.float())

            running_loss += loss.item()

    return running_loss / len(loader)


# training part

train_df = df[df["fold"].isin(cfg.TRAIN_FOLDS)].reset_index(drop=True)
val_df   = df[df["fold"] == cfg.VAL_FOLD].reset_index(drop=True)

class_counts    = train_df["primary_label"].value_counts().to_dict()
sample_weights  = [1.0 / np.sqrt(class_counts[lbl]) for lbl in train_df["primary_label"]]
sampler         = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True,
)

train_dataset = BirdCLEFDataset(train_df, NUM_CLASSES, LABEL2IDX, mode="train")
val_dataset   = BirdCLEFDataset(val_df,   NUM_CLASSES, LABEL2IDX, mode="valid")

train_loader = DataLoader(
    train_dataset,
    batch_size  = cfg.BATCH_SIZE,
    sampler     = sampler,          
    num_workers = cfg.NUM_WORKERS,
    pin_memory  = torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size  = cfg.BATCH_SIZE,
    shuffle     = False,
    num_workers = cfg.NUM_WORKERS,
    pin_memory  = torch.cuda.is_available(),
)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")


model = BirdSEDModel(
    model_name      = cfg.MODEL_NAME,
    num_classes     = NUM_CLASSES,
    pretrained      = False,             
    pretrained_path = cfg.PRETRAINED_WEIGHTS_PATH,
).to(cfg.DEVICE)

criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.EPOCHS * len(train_loader)
)

scaler = torch.amp.GradScaler(device=cfg.AMP_DEVICE)


best_val_loss = float("inf")
history       = []

print(f"\nTraining on {cfg.DEVICE}  |  {cfg.EPOCHS} epochs  |  {NUM_CLASSES} classes")
print("─" * 55)

for epoch in range(1, cfg.EPOCHS + 1):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, scaler, cfg.DEVICE
    )
    val_loss = validate(model, val_loader, criterion, cfg.DEVICE)

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    improved = "*" if val_loss < best_val_loss else ""

    print(
        f"Epoch {epoch:>2}/{cfg.EPOCHS}  "
        f"train={train_loss:.4f}  val={val_loss:.4f}  {improved}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), cfg.TRAINED_MODEL_PATH)

print(f"\nBest val loss : {best_val_loss:.4f}")
print(f"Weights saved : {cfg.TRAINED_MODEL_PATH}")


# onnx 

if ONNX_OK:
    try:
        model.eval()
        dummy = torch.zeros(1, 1, 128, 312).to(cfg.DEVICE)
        torch.onnx.export(
            model,
            dummy,
            cfg.ONNX_PATH,
            export_params=True,
            opset_version=17,
            do_constant_folding=True,
            input_names=["input"],
            output_names=["output"],
            dynamic_axes={
                "input":  {0: "batch_size"},
                "output": {0: "batch_size"},
            },
        )
        print(f"ONNX model saved: {cfg.ONNX_PATH}")
    except Exception as e:
        print(f"[ERROR] ONNX export failed even with packages present: {e}")
else:
    print("[WARN] Skipping ONNX export — dependencies missing.")
meta_path = Path(cfg.OUTPUT_DIR) / "target_columns.json"
with open(meta_path, "w") as f:
    json.dump(TARGET_COLUMNS, f)
print(f"Target columns saved : {meta_path}")



# inference

meta_path = Path(cfg.OUTPUT_DIR) / "target_columns.json"

if meta_path.exists():
    with open(meta_path) as f:
        INF_TARGET_COLUMNS = json.load(f)
    print(f"[INFO] Loaded {len(INF_TARGET_COLUMNS)} target columns from {meta_path}")
else:
    _tax = pd.read_csv(cfg.TAXONOMY_CSV)
    INF_TARGET_COLUMNS = sorted(_tax["primary_label"].astype(str).unique().tolist())
    print(f"[INFO] Derived {len(INF_TARGET_COLUMNS)} target columns from taxonomy.csv")


def build_mel_transforms(cfg: CFG):
    """Returns (mel_transform, db_transform) on CPU."""
    mel = T.MelSpectrogram(
        sample_rate = cfg.TARGET_SR,
        n_fft       = cfg.N_FFT,
        hop_length  = cfg.HOP_LENGTH,
        n_mels      = cfg.N_MELS,
        f_min       = cfg.FMIN,
        f_max       = cfg.FMAX,
    )
    db = T.AmplitudeToDB(stype="power", top_db=80)
    return mel, db


def audio_to_spec(wav_segment: torch.Tensor, mel_t, db_t) -> np.ndarray:
    """Convert a (1, samples) waveform tensor to a normalised ONNX-ready array."""
    spec = db_t(mel_t(wav_segment))        
    spec = (spec - spec.mean()) / (spec.std() + 1e-6)
    return spec.unsqueeze(0).numpy()       


def predict_file(
    path: str,
    session,
    mel_t,
    db_t,
    cfg: CFG,
    n_windows: int = 12,
) -> list:
    """
    Slide a 5-second window over a 60-second soundscape.
    Returns a list of (row_id, prob_array) tuples.
    """
    fname = Path(path).stem

    try:
        wav, sr = torchaudio.load(path)
    except Exception as e:
        print(f"[ERROR] Could not load {path}: {e}")
        return []

    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    if sr != cfg.TARGET_SR:
        wav = T.Resample(orig_freq=sr, new_freq=cfg.TARGET_SR)(wav)

    seg_len      = int(cfg.SEGMENT_SEC * cfg.TARGET_SR)
    input_name   = session.get_inputs()[0].name
    results      = []

    for i in range(n_windows):
        start = i * seg_len
        end   = start + seg_len

        if start >= wav.shape[1]:
            break

        segment = wav[:, start:end]

        if segment.shape[1] < seg_len:
            segment = F.pad(segment, (0, seg_len - segment.shape[1]))

        spec  = audio_to_spec(segment, mel_t, db_t)      
        probs = session.run(None, {input_name: spec})[0]  
        probs = probs[0]                                  

        row_id = f"{fname}_{(i + 1) * 5}"
        results.append((row_id, probs))

    return results


if not ONNX_OK:
    raise RuntimeError(
        "onnxruntime is not installed — cannot run inference. "
        "Install it in a connected Kaggle session and re-run."
    )

if not Path(cfg.ONNX_PATH).exists():
    raise FileNotFoundError(
        f"ONNX model not found at {cfg.ONNX_PATH}. "
        "Run the training block first to generate it."
    )

session = ort.InferenceSession(
    cfg.ONNX_PATH,
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    if torch.cuda.is_available()
    else ["CPUExecutionProvider"],
)
print(f"ONNX session loaded — providers: {session.get_providers()}")
print(f"Input  : {session.get_inputs()[0].shape}")
print(f"Output : {session.get_outputs()[0].shape}")

mel_t, db_t = build_mel_transforms(cfg)
test_files  = sorted(glob.glob(f"{cfg.TEST_DIR}/*.ogg"))
print(f"Test soundscapes found : {len(test_files)}")


if test_files:
    all_rows = []
    for fp in tqdm(test_files, desc="Inference"):
        rows = predict_file(fp, session, mel_t, db_t, cfg)
        all_rows.extend(rows)

    submission_df = pd.DataFrame(
        [{"row_id": r, **dict(zip(INF_TARGET_COLUMNS, p))} for r, p in all_rows]
    )
    submission_df.to_csv("submission.csv", index=False)
    print(f"submission.csv saved — {len(submission_df):,} rows × {len(INF_TARGET_COLUMNS) + 1} cols")

else:
    print("[INFO] No test soundscapes found — generating dummy submission.")
    sample_sub = pd.read_csv(cfg.SAMPLE_SUB)

    for col in INF_TARGET_COLUMNS:
        if col in sample_sub.columns:
            sample_sub[col] = 0.5

    sample_sub.to_csv("submission.csv", index=False)
    print(f"Dummy submission.csv saved — {len(sample_sub):,} rows")

print("\nDone.")

PyTorch    : 2.10.0+cu128
torchaudio : 2.10.0+cu128
CUDA       : True
ONNX ready : True
Device : cuda
AMP    : cuda
Total samples  : 35,549
Unique species : 234
Fold counts    :
fold
0    7110
1    7110
2    7110
3    7110
4    7109
Train batches : 889
Val   batches : 223

Training on cuda  |  12 epochs  |  234 classes
───────────────────────────────────────────────────────


  train:   0%|          | 0/889 [00:00<?, ?it/s]

    val:   0%|          | 0/223 [00:00<?, ?it/s]

Epoch  1/12  train=0.0019  val=0.0015  *


  train:   0%|          | 0/889 [00:00<?, ?it/s]

...

[torch.onnx] Obtain model graph for `BirdSEDModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BirdSEDModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Runtim

Applied 220 of general pattern rewrite rules.
ONNX model saved: /kaggle/working/bird_model.onnx
Target columns saved : /kaggle/working/target_columns.json
[INFO] Loaded 234 target columns from /kaggle/working/target_columns.json
ONNX session loaded — providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
Input  : ['batch_size', 1, 128, 312]
Output : ['batch_size', 234]
Test soundscapes found : 0
[INFO] No test soundscapes found — generating dummy submission.
Dummy submission.csv saved — 3 rows

Done.
